# [12.3] Mini VLM from Scratch

> **ARENA extension note.** Original ARENA is unchanged. This section builds a visual-prefix causal decoder one component at a time; the complete architecture is never provided before its exercises.

By the end of this notebook, you will have shown that a frozen patch encoder, learned connector, and two-layer causal decoder solve held-out visual questions because joint VQA succeeds while text-only, image-only, and shuffled-image controls fail, and counterfactual object-token patches flip answers while matched non-object patches do not.

## Core Question

Can you build the entire image-to-answer path yourself, then show causally that its final answer reads the relevant visual prefix rather than a question prior?

**Common bug:** implementing a classifier over pooled image features and calling it a VLM. Here the model has explicit image tokens, a question token, a causal mask, inspectable per-layer value streams, and an answer loss at the final token.

## Learning Objectives

You will implement, in dependency order:

1. a frozen patch encoder which emits RGB-plus-occupancy tokens;
2. detached visual-token caching;
3. a learned connector into the decoder residual width;
4. insertion of visual tokens before the question token;
5. exact token-row replacement;
6. a causal multi-head self-attention block;
7. the complete two-layer MiniVLM decoder;
8. answer cross entropy, accuracy, and counterfactual margin;
9. image-box to token-index geometry;
10. an exact additive patching oracle;
11. training and modality baselines; and
12. trained object/background/random/full-sequence activation patching.


In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch as t
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

GT_TIER = "GT-0"
EXERCISE_ID = "12_3_mini_vlm_from_scratch"
DIFFICULTY = 4
IMPORTANCE = 5
EXPECTED_RUNTIME = "75-105 minutes; reference CPU training takes seconds"
REQUIRES_GPU = True

chapter = "chapter12_vlm_interpretability"
section = "part3_mini_vlm_from_scratch"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
assets_dir = root_dir / chapter / "instructions" / "assets"
for path in (root_dir, exercises_dir):
    if str(path) not in sys.path:
        sys.path.append(str(path))

import part3_mini_vlm_from_scratch.tests as tests
import part3_mini_vlm_from_scratch.solutions as reference

COLORS = reference.COLORS
SHAPES = reference.SHAPES
ANSWER_VOCAB = reference.ANSWER_VOCAB
ANSWER_TO_ID = reference.ANSWER_TO_ID
QUESTION_TYPES = reference.QUESTION_TYPES
QUESTION_TO_ID = reference.QUESTION_TO_ID
UNKNOWN_QUESTION_ID = reference.UNKNOWN_QUESTION_ID
IMAGE_SIZE = reference.IMAGE_SIZE
PATCH_GRID = reference.PATCH_GRID
OBJECT_SIZE = reference.OBJECT_SIZE
VISION_FEATURE_DIM = reference.VISION_FEATURE_DIM
DEFAULT_D_MODEL = reference.DEFAULT_D_MODEL
DEFAULT_SEED = reference.DEFAULT_SEED
POSITION_ANCHORS = reference.POSITION_ANCHORS
TRAIN_STYLE = reference.TRAIN_STYLE
HELDOUT_STYLE = reference.HELDOUT_STYLE

VQAExample = reference.VQAExample
VQABatch = reference.VQABatch
MiniVLMTrainingResult = reference.MiniVLMTrainingResult
render_controlled_scene = reference.render_controlled_scene
build_vqa_batch = reference.build_vqa_batch
same_size_random_region_indices = reference.same_size_random_region_indices
_next_value = reference._next_value


## Cold Open

The clean image is a muted red square; the counterfactual is the same scene with a blue square. The question remains *What color is the object?* Before training anything, predict which visual rows must change to flip the answer.


In [ ]:
clean_image, clean_bbox = render_controlled_scene("red", "square", "top", style=HELDOUT_STYLE)
corrupt_image, _ = render_controlled_scene("blue", "square", "top", style=HELDOUT_STYLE)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
for ax, image, title in zip(axes, (clean_image, corrupt_image), ("clean: red square", "counterfactual: blue square")):
    ax.imshow(image.permute(1, 2, 0)); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()
print("object bbox:", clean_bbox, "| fixed question: What color is the object?")


### Exercise 1 - Build the frozen patch encoder

Turn each image into a `6 x 6` token grid. Every token contains patch mean RGB and exact non-white occupancy. This is deliberately inspectable ground truth, not a learned vision model.


In [ ]:
class FrozenPatchEncoder(nn.Module):
    def __init__(self, *, grid_size: int = PATCH_GRID):
        super().__init__()
        self.grid_size = grid_size

    @property
    def num_tokens(self) -> int:
        return self.grid_size * self.grid_size

    @property
    def feature_dim(self) -> int:
        return VISION_FEATURE_DIM

    def forward(self, images: t.Tensor) -> t.Tensor:
        # Rearrange pixels into patches, then emit mean RGB and non-white occupancy.
        raise NotImplementedError()


In [ ]:
tests.test_frozen_patch_encoder_extracts_rgb_and_occupancy(FrozenPatchEncoder)


<details>
    <summary>Expected output</summary>

    Output shape is `[1, 36, 4]`; pure object/background occupancy is exactly `1/0`.

    </details>

    <details>
    <summary>Help - locate the invariant</summary>

    Reshape into grid row, patch row, grid column, patch column; then average spatial axes.

    </details>

    <details>
    <summary>Solution</summary>

    ```python
    class FrozenPatchEncoder(nn.Module):
"""Frozen pixel-patch encoder used as the mini VLM vision tower."""

def __init__(self, *, grid_size: int = PATCH_GRID):
    super().__init__()
    self.grid_size = grid_size

@property
def num_tokens(self) -> int:
    return self.grid_size * self.grid_size

@property
def feature_dim(self) -> int:
    return VISION_FEATURE_DIM

def forward(self, images: t.Tensor) -> t.Tensor:
    if images.ndim != 4 or images.shape[1] != 3:
        raise ValueError("images must have shape (batch, 3, height, width).")
    batch, channels, height, width = images.shape
    if height != width or height % self.grid_size != 0:
        raise ValueError("image height/width must be square and divisible by grid_size.")
    patch = height // self.grid_size
    patches = images.float().reshape(
        batch,
        channels,
        self.grid_size,
        patch,
        self.grid_size,
        patch,
    )
    patches = patches.permute(0, 2, 4, 1, 3, 5)
    mean_rgb = patches.mean(dim=(-1, -2))
    nonwhite = (patches < 0.98).any(dim=3).float().mean(dim=(-1, -2))
    tokens = t.cat(
        [
            mean_rgb,
            nonwhite.unsqueeze(-1),
        ],
        dim=-1,
    )
    return tokens.reshape(batch, self.num_tokens, self.feature_dim)
    ```

    </details>

    <details>
    <summary>Interpretation</summary>

    The encoder exposes where image evidence enters. Later patching claims refer to these exact rows.

    </details>


### Exercise 2 - Cache frozen visual tokens

Run the encoder once without gradients and detach the result so decoder training cannot silently alter the visual representation.


In [ ]:
def encode_visual_token_cache(encoder: FrozenPatchEncoder, images: t.Tensor, *, device: str | t.device = 'cpu') -> t.Tensor:
    raise NotImplementedError()


In [ ]:
tests.test_visual_token_cache_detaches_and_preserves_patch_grid(encode_visual_token_cache, FrozenPatchEncoder)


<details>
        <summary>Expected output</summary>

        The cache has `[120, 36, 4]`, has no gradient history, and contains occupied and empty patches.

        </details>

        <details>
        <summary>Help - locate the invariant</summary>

        Move both encoder and images to the target device inside `no_grad`, then detach.

        </details>

        <details>
        <summary>Solution</summary>

        ```python
        def encode_visual_token_cache(
    encoder: FrozenPatchEncoder,
    images: t.Tensor,
    *,
    device: str | t.device = "cpu",
) -> t.Tensor:
    """Encode images once and detach the visual tokens for fast VLM training."""

    target_device = t.device(device)
    encoder = encoder.to(target_device)
    with t.no_grad():
        cache = encoder(images.to(target_device))
    return cache.detach()
        ```

        </details>

        <details>
        <summary>Interpretation</summary>

        Caching makes the frozen-versus-learned boundary explicit and keeps every later intervention reproducible.

        </details>


### Exercise 3 - Learn the visual connector

Map each four-dimensional frozen patch token into the decoder residual width using a learned projection and layer normalization.


In [ ]:
class VisualConnector(nn.Module):
    def __init__(self, vision_feature_dim: int, d_model: int):
        super().__init__()
        raise NotImplementedError()

    def forward(self, visual_cache: t.Tensor) -> t.Tensor:
        raise NotImplementedError()


In [ ]:
tests.test_visual_connector_projects_every_patch(VisualConnector)


<details>
    <summary>Expected output</summary>

    `[3, 36, 4]` becomes `[3, 36, 24]` without pooling or changing token order.

    </details>

    <details>
    <summary>Help - locate the invariant</summary>

    Apply the same linear map independently to every patch, then normalize only the feature axis.

    </details>

    <details>
    <summary>Solution</summary>

    ```python
    class VisualConnector(nn.Module):
"""Map frozen vision features into the decoder residual-stream width."""

def __init__(self, vision_feature_dim: int, d_model: int):
    super().__init__()
    self.proj = nn.Linear(vision_feature_dim, d_model)
    self.norm = nn.LayerNorm(d_model)

def forward(self, visual_cache: t.Tensor) -> t.Tensor:
    if visual_cache.ndim != 3:
        raise ValueError("visual_cache must have shape (batch, tokens, features).")
    return self.norm(self.proj(visual_cache.float()))
    ```

    </details>

    <details>
    <summary>Interpretation</summary>

    The connector is the learned bridge between vision features and language-decoder states.

    </details>


### Exercise 4 - Insert image tokens before the question token

Project the visual cache and append one embedded question token. The final sequence position will be trained to predict the answer.


In [ ]:
def build_multimodal_sequence(model, visual_cache: t.Tensor, question_ids: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


In [ ]:
tests.test_multimodal_sequence_has_visual_prefix_and_question_token(build_multimodal_sequence)


<details>
        <summary>Expected output</summary>

        Three examples produce `[3, 37, 24]`: 36 visual positions followed by one question position.

        </details>

        <details>
        <summary>Help - locate the invariant</summary>

        Embed question IDs, add a length-one sequence axis, and concatenate after the visual prefix.

        </details>

        <details>
        <summary>Solution</summary>

        ```python
        def build_multimodal_sequence(
    model: MiniVLM,
    visual_cache: t.Tensor,
    question_ids: t.Tensor,
) -> t.Tensor:
    """Project visual tokens and append the question token."""

    visual_tokens = model.visual_connector(visual_cache)
    question_tokens = model.question_embed(question_ids.to(visual_tokens.device)).unsqueeze(1)
    return t.cat([visual_tokens, question_tokens], dim=1)
        ```

        </details>

        <details>
        <summary>Interpretation</summary>

        This explicit sequence is what makes causal masking meaningful: the answer position can read image tokens, while image tokens cannot read the later question.

        </details>


### Exercise 5 - Replace selected token rows exactly

Clone the clean tensor and replace only unique in-range rows with counterfactual rows. Write both the cache-specific and general replacement functions.


In [ ]:
def patch_visual_tokens(clean_cache: t.Tensor, corrupt_cache: t.Tensor, patch_indices: tuple[int, ...] | list[int]) -> t.Tensor:
    raise NotImplementedError()

def _replace_token_rows(values: t.Tensor, patch_indices: tuple[int, ...] | list[int], replacement_rows: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


In [ ]:
tests.test_patch_visual_tokens_replaces_only_selected_rows(patch_visual_tokens)


<details>
        <summary>Expected output</summary>

        Rows 1 and 3 change; all unselected rows and the original clean tensor remain unchanged.

        </details>

        <details>
        <summary>Help - locate the invariant</summary>

        Validate rank, shape, nonempty unique indices, and bounds before indexed assignment into a clone.

        </details>

        <details>
        <summary>Solution</summary>

        ```python
        def patch_visual_tokens(
    clean_cache: t.Tensor,
    corrupt_cache: t.Tensor,
    patch_indices: tuple[int, ...] | list[int],
) -> t.Tensor:
    """Return ``clean_cache`` with selected token rows replaced by ``corrupt_cache``."""

    if clean_cache.shape != corrupt_cache.shape:
        raise ValueError("clean_cache and corrupt_cache must have the same shape.")
    if clean_cache.ndim != 3:
        raise ValueError("visual caches must have shape (batch, tokens, features).")
    if len(patch_indices) == 0:
        raise ValueError("patch_indices must be nonempty.")
    index = t.tensor(tuple(int(i) for i in patch_indices), device=clean_cache.device)
    if index.min() < 0 or index.max() >= clean_cache.shape[1]:
        raise ValueError("patch index out of range.")
    if len(set(index.detach().cpu().tolist())) != index.numel():
        raise ValueError("patch_indices must be unique.")
    patched = clean_cache.clone()
    patched[:, index] = corrupt_cache[:, index].to(
        device=patched.device,
        dtype=patched.dtype,
    )
    return patched


def _replace_token_rows(
    values: t.Tensor,
    patch_indices: tuple[int, ...] | list[int],
    replacement_rows: t.Tensor,
) -> t.Tensor:
    index = t.tensor(tuple(int(i) for i in patch_indices), device=values.device)
    if replacement_rows.ndim != 3:
        raise ValueError("replacement_rows must have shape (batch, patched_tokens, dim).")
    if replacement_rows.shape[1] != index.numel():
        raise ValueError("replacement row count must match patch_indices length.")
    if replacement_rows.shape[0] == 1 and values.shape[0] != 1:
        replacement_rows = replacement_rows.expand(values.shape[0], -1, -1)
    if replacement_rows.shape[0] != values.shape[0]:
        raise ValueError("replacement batch size must be 1 or match values batch size.")
    patched = values.clone()
    patched[:, index] = replacement_rows.to(device=values.device, dtype=values.dtype)
    return patched
        ```

        </details>

        <details>
        <summary>Interpretation</summary>

        Token replacement is the causal primitive. If it changes extra rows, every localization result is invalid.

        </details>


### Exercise 6 - Implement one causal decoder block

Write pre-norm multi-head self-attention, an upper-triangular causal mask, residual output projection, and an MLP. Return the unpatched value stream and attention for later interventions.


In [ ]:
class CausalDecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        raise NotImplementedError()

    def forward(
        self,
        sequence: t.Tensor,
        *,
        replacement_values: t.Tensor | None = None,
        patch_indices: tuple[int, ...] = (),
    ) -> tuple[t.Tensor, t.Tensor, t.Tensor]:
        raise NotImplementedError()


In [ ]:
tests.test_causal_decoder_block_cannot_read_future_tokens(CausalDecoderBlock)


<details>
    <summary>Expected output</summary>

    Changing only the final token leaves every earlier position invariant; attention above the diagonal is exactly zero.

    </details>

    <details>
    <summary>Help - locate the invariant</summary>

    Split `[B, S, D]` into `[B, H, S, Dh]`; mask keys strictly to the right before softmax.

    </details>

    <details>
    <summary>Solution</summary>

    ```python
    class CausalDecoderBlock(nn.Module):
"""One pre-norm causal self-attention block with an inspectable value stream."""

def __init__(self, d_model: int, num_heads: int):
    super().__init__()
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads.")
    self.d_model = d_model
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads
    self.ln1 = nn.LayerNorm(d_model)
    self.q_proj = nn.Linear(d_model, d_model)
    self.k_proj = nn.Linear(d_model, d_model)
    self.v_proj = nn.Linear(d_model, d_model)
    self.out_proj = nn.Linear(d_model, d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.mlp = nn.Sequential(
        nn.Linear(d_model, 4 * d_model),
        nn.GELU(),
        nn.Linear(4 * d_model, d_model),
    )

def forward(
    self,
    sequence: t.Tensor,
    *,
    replacement_values: t.Tensor | None = None,
    patch_indices: tuple[int, ...] = (),
) -> tuple[t.Tensor, t.Tensor, t.Tensor]:
    batch, seq_len, _ = sequence.shape
    normalized = self.ln1(sequence)

    def split_heads(values: t.Tensor) -> t.Tensor:
        return values.reshape(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

    queries = split_heads(self.q_proj(normalized))
    keys = split_heads(self.k_proj(normalized))
    values = self.v_proj(normalized)
    cached_values = values.detach()
    if replacement_values is not None:
        values = _replace_token_rows(values, patch_indices, replacement_values)
    values_by_head = split_heads(values)
    scores = queries @ keys.transpose(-1, -2) / self.head_dim**0.5
    causal_mask = t.triu(
        t.ones(seq_len, seq_len, dtype=t.bool, device=sequence.device),
        diagonal=1,
    )
    attention = scores.masked_fill(causal_mask, float("-inf")).softmax(dim=-1)
    context = (attention @ values_by_head).transpose(1, 2).reshape(batch, seq_len, self.d_model)
    sequence = sequence + self.out_proj(context)
    sequence = sequence + self.mlp(self.ln2(sequence))
    return sequence, cached_values, attention.detach()
    ```

    </details>

    <details>
    <summary>Interpretation</summary>

    Causality is tested behaviorally, not inferred from a variable named `mask`.

    </details>


### Exercise 7 - Assemble the MiniVLM

Combine your connector, question embedding, two causal blocks, final normalization, and answer head. Support cache, projected, and per-layer value patches without changing the clean question.


In [ ]:
class MiniVLM(nn.Module):
    def __init__(
        self,
        *,
        vision_feature_dim: int = VISION_FEATURE_DIM,
        d_model: int = DEFAULT_D_MODEL,
        num_visual_tokens: int = PATCH_GRID * PATCH_GRID,
        num_layers: int = 2,
        num_heads: int = 4,
        num_answers: int = len(ANSWER_VOCAB),
    ):
        super().__init__()
        raise NotImplementedError()

    def forward(
        self,
        visual_cache: t.Tensor,
        question_ids: t.Tensor,
        *,
        ablate_question: bool = False,
        patch_spec: dict[str, object] | None = None,
        return_cache: bool = False,
    ):
        raise NotImplementedError()


In [ ]:
tests.test_mini_vlm_uses_visual_prefix_causal_attention(MiniVLM)


<details>
    <summary>Expected output</summary>

    The model returns `[batch, 7]` answer logits and caches `value_0`, `value_1`, and final-position attention.

    </details>

    <details>
    <summary>Help - locate the invariant</summary>

    Build the 37-token sequence once. Apply patches at their named stage, then read only the normalized final token.

    </details>

    <details>
    <summary>Solution</summary>

    ```python
    class MiniVLM(nn.Module):
"""Tiny visual-prefix causal decoder which predicts from the final question token."""

def __init__(
    self,
    *,
    vision_feature_dim: int = VISION_FEATURE_DIM,
    d_model: int = DEFAULT_D_MODEL,
    num_visual_tokens: int = PATCH_GRID * PATCH_GRID,
    num_layers: int = 2,
    num_heads: int = 4,
    num_answers: int = len(ANSWER_VOCAB),
):
    super().__init__()
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads.")
    self.num_visual_tokens = num_visual_tokens
    self.d_model = d_model
    self.num_layers = num_layers
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads
    self.visual_connector = VisualConnector(vision_feature_dim, d_model)
    self.question_embed = nn.Embedding(len(QUESTION_TYPES) + 1, d_model)
    self.decoder_blocks = nn.ModuleList(
        [CausalDecoderBlock(d_model, num_heads) for _ in range(num_layers)]
    )
    self.final_norm = nn.LayerNorm(d_model)
    self.answer_head = nn.Linear(d_model, num_answers)
    with t.no_grad():
        self.question_embed.weight[UNKNOWN_QUESTION_ID].zero_()

def forward(
    self,
    visual_cache: t.Tensor,
    question_ids: t.Tensor,
    *,
    ablate_question: bool = False,
    patch_spec: dict[str, object] | None = None,
    return_cache: bool = False,
) -> t.Tensor | tuple[t.Tensor, dict[str, t.Tensor]]:
    if visual_cache.ndim != 3:
        raise ValueError("visual_cache must have shape (batch, visual_tokens, features).")
    if visual_cache.shape[1] != self.num_visual_tokens:
        raise ValueError("visual_cache has the wrong number of visual tokens.")
    if question_ids.ndim != 1 or question_ids.shape[0] != visual_cache.shape[0]:
        raise ValueError("question_ids must have shape (batch,).")

    stage = None if patch_spec is None else str(patch_spec["stage"])
    patch_indices = () if patch_spec is None else tuple(patch_spec["indices"])  # type: ignore[arg-type]
    replacement = None if patch_spec is None else patch_spec["values"]  # type: ignore[index]

    if stage == "cache":
        visual_cache = _replace_token_rows(visual_cache, patch_indices, replacement)  # type: ignore[arg-type]

    visual_tokens = self.visual_connector(visual_cache)
    cache: dict[str, t.Tensor] = {}
    if return_cache:
        cache["cache"] = visual_cache.detach()
        cache["projected"] = visual_tokens.detach()
    if stage == "projected":
        visual_tokens = _replace_token_rows(visual_tokens, patch_indices, replacement)  # type: ignore[arg-type]

    question_tokens = self.question_embed(question_ids.to(visual_cache.device))
    if ablate_question:
        question_tokens = t.zeros_like(question_tokens)
    sequence = t.cat([visual_tokens, question_tokens.unsqueeze(1)], dim=1)
    for layer_index, block in enumerate(self.decoder_blocks):
        layer_stage = f"value_{layer_index}"
        sequence, values, attention = block(
            sequence,
            replacement_values=replacement if stage == layer_stage else None,  # type: ignore[arg-type]
            patch_indices=patch_indices,
        )
        if return_cache:
            cache[layer_stage] = values[:, : self.num_visual_tokens]
            cache[f"attention_{layer_index}"] = attention[:, :, -1, :].detach()

    answer_state = self.final_norm(sequence[:, -1])
    logits = self.answer_head(answer_state)
    if return_cache:
        return logits, cache
    return logits
    ```

    </details>

    <details>
    <summary>Interpretation</summary>

    The answer is decoded from a causal state that can attend over every earlier visual token.

    </details>


### Exercise 8 - Implement answer metrics and the training loss

Write exact-match accuracy, target-minus-counterfactual margin, and cross entropy on the final answer logits. Confirm gradients reach the visual connector.


In [ ]:
def vqa_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()

def answer_margin(logits: t.Tensor, target_labels: t.Tensor, counterfactual_labels: t.Tensor) -> t.Tensor:
    raise NotImplementedError()

def mini_vlm_loss(model: MiniVLM, visual_cache: t.Tensor, question_ids: t.Tensor, labels: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


In [ ]:
tests.test_vqa_accuracy_and_answer_margin(vqa_accuracy, answer_margin)
tests.test_mini_vlm_loss_is_next_answer_cross_entropy(mini_vlm_loss, MiniVLM)


<details>
        <summary>Expected output</summary>

        The exact metric oracle returns accuracy `0.5` and margins `[3, -1]`; loss gradients reach the connector.

        </details>

        <details>
        <summary>Help - locate the invariant</summary>

        Use `gather` for margins and call the model without `return_cache` for answer cross entropy.

        </details>

        <details>
        <summary>Solution</summary>

        ```python
        def vqa_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    """Return exact-match VQA accuracy."""

    if logits.ndim != 2:
        raise ValueError("logits must have shape (batch, answers).")
    if labels.ndim != 1 or labels.shape[0] != logits.shape[0]:
        raise ValueError("labels must have shape (batch,).")
    return float((logits.argmax(dim=-1) == labels.to(logits.device)).float().mean().item())


def answer_margin(
    logits: t.Tensor,
    target_labels: t.Tensor,
    counterfactual_labels: t.Tensor,
) -> t.Tensor:
    """Return target-answer logit minus counterfactual-answer logit."""

    if target_labels.shape != counterfactual_labels.shape:
        raise ValueError("target and counterfactual labels must have matching shapes.")
    target = logits.gather(1, target_labels.to(logits.device).reshape(-1, 1)).squeeze(1)
    counter = logits.gather(
        1,
        counterfactual_labels.to(logits.device).reshape(-1, 1),
    ).squeeze(1)
    return target - counter


def mini_vlm_loss(
    model: MiniVLM,
    visual_cache: t.Tensor,
    question_ids: t.Tensor,
    labels: t.Tensor,
) -> t.Tensor:
    """Return next-answer cross entropy from the final causal question position."""

    logits = model(visual_cache, question_ids)
    if not isinstance(logits, t.Tensor):
        raise TypeError("MiniVLM must return logits when return_cache=False.")
    return F.cross_entropy(logits, labels.to(logits.device))
        ```

        </details>

        <details>
        <summary>Interpretation</summary>

        Accuracy tests prediction; the signed margin is the more sensitive causal intervention metric.

        </details>


### Exercise 9 - Map pixels to visual-token indices

Enumerate every patch rectangle and return the row-major token indices whose pixel area overlaps the object's bounding box.


In [ ]:
def patch_indices_from_bbox(bbox: tuple[int, int, int, int], *, image_size: int = IMAGE_SIZE, grid_size: int = PATCH_GRID) -> tuple[int, ...]:
    raise NotImplementedError()


In [ ]:
tests.test_patch_indices_from_bbox_matches_known_grid(patch_indices_from_bbox)


<details>
        <summary>Expected output</summary>

        The centered 24-pixel square overlaps the exact 4 x 4 token block `(7, ..., 28)`.

        </details>

        <details>
        <summary>Help - locate the invariant</summary>

        Two rectangles overlap only when both axis intervals have positive intersection.

        </details>

        <details>
        <summary>Solution</summary>

        ```python
        def patch_indices_from_bbox(
    bbox: tuple[int, int, int, int],
    *,
    image_size: int = IMAGE_SIZE,
    grid_size: int = PATCH_GRID,
) -> tuple[int, ...]:
    """Return patch-token indices whose image patches overlap ``bbox``."""

    x1, y1, x2, y2 = bbox
    if not (0 <= x1 < x2 <= image_size and 0 <= y1 < y2 <= image_size):
        raise ValueError("bbox must lie inside the image with positive area.")
    if image_size % grid_size != 0:
        raise ValueError("image_size must be divisible by grid_size.")
    patch = image_size // grid_size
    indices: list[int] = []
    for row in range(grid_size):
        patch_y1 = row * patch
        patch_y2 = patch_y1 + patch
        for col in range(grid_size):
            patch_x1 = col * patch
            patch_x2 = patch_x1 + patch
            overlaps = max(x1, patch_x1) < min(x2, patch_x2) and max(y1, patch_y1) < min(
                y2, patch_y2
            )
            if overlaps:
                indices.append(row * grid_size + col)
    if not indices:
        raise ValueError("bbox does not overlap any patch token.")
    return tuple(indices)
        ```

        </details>

        <details>
        <summary>Interpretation</summary>

        This geometry connects a visible object region to the hidden rows you intervene on.

        </details>


### Exercise 10 - Establish exact patching ground truth

Put all red-vs-blue evidence in known object rows of an additive token game. Patch object, background, and same-size random rows before training the neural model.


In [ ]:
def toy_ground_truth_patch_report(*, grid_size: int = PATCH_GRID, target_answer: str = 'red', counterfactual_answer: str = 'blue') -> dict[str, object]:
    raise NotImplementedError()


In [ ]:
toy_report = toy_ground_truth_patch_report()
tests.test_toy_ground_truth_patch_report_has_exact_controls(toy_ground_truth_patch_report)
display(pd.DataFrame([{k: v for k, v in toy_report.items() if k.endswith("margin")}]))


<details>
        <summary>Expected output</summary>

        Only the object patch makes the signed answer margin negative; matched non-object patches preserve it.

        </details>

        <details>
        <summary>Help - locate the invariant</summary>

        Construct clean and corrupt token contributions explicitly, sum over token rows, and compare answer columns.

        </details>

        <details>
        <summary>Solution</summary>

        ```python
        def toy_ground_truth_patch_report(
    *,
    grid_size: int = PATCH_GRID,
    target_answer: str = "red",
    counterfactual_answer: str = "blue",
) -> dict[str, object]:
    """Exact oracle showing why object patching should flip this toy game."""

    num_tokens = grid_size * grid_size
    object_indices = patch_indices_from_bbox(
        POSITION_ANCHORS["center"]
        + (
            POSITION_ANCHORS["center"][0] + OBJECT_SIZE,
            POSITION_ANCHORS["center"][1] + OBJECT_SIZE,
        ),
        grid_size=grid_size,
    )
    background_indices = same_size_random_region_indices(
        object_indices,
        num_tokens=num_tokens,
        seed=0,
    )
    random_indices = same_size_random_region_indices(
        object_indices,
        num_tokens=num_tokens,
        seed=1,
    )
    target = ANSWER_TO_ID[target_answer]
    counter = ANSWER_TO_ID[counterfactual_answer]
    clean = t.zeros(1, num_tokens, len(ANSWER_VOCAB))
    corrupt = t.zeros_like(clean)
    clean[:, object_indices, target] = 1.0
    clean[:, object_indices, counter] = -0.25
    corrupt[:, object_indices, counter] = 1.0
    corrupt[:, object_indices, target] = -0.25

    def margin(cache: t.Tensor) -> float:
        logits = cache.sum(dim=1)
        return float((logits[:, target] - logits[:, counter]).item())

    object_patched = patch_visual_tokens(clean, corrupt, object_indices)
    background_patched = patch_visual_tokens(clean, corrupt, background_indices)
    random_patched = patch_visual_tokens(clean, corrupt, random_indices)
    return {
        "claim_scope": "exact_toy_visual_token_contribution_game",
        "object_indices": object_indices,
        "background_indices": background_indices,
        "random_indices": random_indices,
        "clean_margin": margin(clean),
        "corrupt_margin": margin(corrupt),
        "object_patch_margin": margin(object_patched),
        "background_patch_margin": margin(background_patched),
        "random_patch_margin": margin(random_patched),
        "object_patch_flips": margin(object_patched) < 0,
        "background_patch_preserves": margin(background_patched) > 0,
        "random_patch_preserves": margin(random_patched) > 0,
        "object_beats_background": margin(background_patched) - margin(object_patched) > 5.0,
    }
        ```

        </details>

        <details>
        <summary>Interpretation</summary>

        The exact oracle defines what the trained-model intervention should recover and what each control rules out.

        </details>


### Exercise 11 - Train and falsify the MiniVLM

Train on 120 solid scenes, evaluate on 120 muted-style scenes, and compare joint inference with zero-visual, zero-question, and shuffled-visual controls.


In [ ]:
def train_mini_vlm(*, steps: int = 320, lr: float = 3e-3, seed: int = DEFAULT_SEED, device: str | t.device = 'cpu') -> MiniVLMTrainingResult:
    raise NotImplementedError()

def modality_baseline_report(training: MiniVLMTrainingResult) -> dict[str, float]:
    raise NotImplementedError()


In [ ]:
training = train_mini_vlm(steps=260, device="cpu")
baselines = modality_baseline_report(training)
tests.check_training_and_baselines(training, baselines)
display(pd.DataFrame([baselines]))
fig, ax = plt.subplots(figsize=(5, 3)); ax.plot(training.losses); ax.set(title="MiniVLM answer loss", xlabel="checkpoint", ylabel="cross entropy"); plt.show()


<details>
        <summary>Expected output</summary>

        Train and muted-style held-out accuracy reach `1.0`; text-only, image-only, and shuffled-visual accuracy stay below the lock thresholds.

        </details>

        <details>
        <summary>Help - locate the invariant</summary>

        Cache both splits, optimize only the MiniVLM parameters, then evaluate each ablation on the same held-out labels.

        </details>

        <details>
        <summary>Solution</summary>

        ```python
        def train_mini_vlm(
    *,
    steps: int = 320,
    lr: float = 3e-3,
    seed: int = DEFAULT_SEED,
    device: str | t.device = "cpu",
) -> MiniVLMTrainingResult:
    """Train on solid scenes and evaluate on the disjoint muted style."""

    target_device = t.device(device)
    if target_device.type == "cpu" and t.get_num_threads() > 4:
        t.set_num_threads(4)
    t.manual_seed(seed)
    encoder = FrozenPatchEncoder().to(target_device)
    train_batch = build_vqa_batch("train")
    heldout_batch = build_vqa_batch("heldout")
    train_cache = encode_visual_token_cache(encoder, train_batch.images, device=target_device)
    heldout_cache = encode_visual_token_cache(encoder, heldout_batch.images, device=target_device)
    train_questions = train_batch.question_ids.to(target_device)
    train_labels = train_batch.labels.to(target_device)
    heldout_questions = heldout_batch.question_ids.to(target_device)
    heldout_labels = heldout_batch.labels.to(target_device)

    model = MiniVLM().to(target_device)
    optimizer = t.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    losses: list[float] = []
    for step in range(steps):
        loss = mini_vlm_loss(model, train_cache, train_questions, train_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % max(1, steps // 80) == 0 or step == steps - 1:
            losses.append(float(loss.detach().cpu().item()))

    model.eval()
    with t.no_grad():
        train_logits = model(train_cache, train_questions)
        heldout_logits = model(heldout_cache, heldout_questions)
    assert isinstance(train_logits, t.Tensor)
    assert isinstance(heldout_logits, t.Tensor)
    return MiniVLMTrainingResult(
        model=model,
        encoder=encoder,
        train_batch=train_batch,
        heldout_batch=heldout_batch,
        train_cache=train_cache.detach(),
        heldout_cache=heldout_cache.detach(),
        losses=tuple(losses),
        train_accuracy=vqa_accuracy(train_logits, train_labels),
        heldout_accuracy=vqa_accuracy(heldout_logits, heldout_labels),
    )


def modality_baseline_report(training: MiniVLMTrainingResult) -> dict[str, float]:
    """Compare joint VQA to text-only, image-only, and random-visual baselines."""

    model = training.model
    cache = training.heldout_cache
    question_ids = training.heldout_batch.question_ids.to(cache.device)
    labels = training.heldout_batch.labels.to(cache.device)
    generator = t.Generator(device="cpu").manual_seed(DEFAULT_SEED)
    permutation = t.randperm(cache.shape[0], generator=generator).to(cache.device)
    with t.no_grad():
        joint = model(cache, question_ids)
        text_only = model(t.zeros_like(cache), question_ids)
        image_only = model(cache, question_ids, ablate_question=True)
        random_visual = model(cache[permutation], question_ids)
    assert isinstance(joint, t.Tensor)
    assert isinstance(text_only, t.Tensor)
    assert isinstance(image_only, t.Tensor)
    assert isinstance(random_visual, t.Tensor)
    return {
        "joint_accuracy": vqa_accuracy(joint, labels),
        "text_only_accuracy": vqa_accuracy(text_only, labels),
        "image_only_accuracy": vqa_accuracy(image_only, labels),
        "random_visual_accuracy": vqa_accuracy(random_visual, labels),
    }
        ```

        </details>

        <details>
        <summary>Interpretation</summary>

        High joint accuracy is not grounding evidence until modality controls remove either the pixels or the question.

        </details>


### Exercise 12 - Patch the trained causal computation

For a clean/counterfactual pair, replace object, background, random, or full visual rows at the cache, projected, and per-layer value stages. Sweep every token to build the causal heatmap.


In [ ]:
def build_single_example(*, color: str, shape: str, position: str, question_type: str, style: str = HELDOUT_STYLE, split: str = 'patch') -> VQABatch:
    raise NotImplementedError()

def _patch_replacement_for_stage(model: MiniVLM, corrupt_cache: t.Tensor, question_ids: t.Tensor, patch_indices: tuple[int, ...], stage: str) -> t.Tensor:
    raise NotImplementedError()

def forward_with_visual_patch(model: MiniVLM, clean_cache: t.Tensor, corrupt_cache: t.Tensor, question_ids: t.Tensor, patch_indices: tuple[int, ...], *, stage: str = 'cache') -> t.Tensor:
    raise NotImplementedError()

def trained_patch_report(training: MiniVLMTrainingResult, *, clean_color: str = 'red', corrupt_color: str = 'blue', clean_shape: str = 'square', corrupt_shape: str = 'square', position: str = 'top', question_type: str = 'color', stage: str = 'cache') -> dict[str, object]:
    raise NotImplementedError()

def patching_effect_heatmap(training: MiniVLMTrainingResult, *, stages: tuple[str, ...] = ('cache', 'projected', 'value_0', 'value_1')) -> dict[str, object]:
    raise NotImplementedError()


In [ ]:
color_patch = trained_patch_report(training, stage="cache")
shape_patch = trained_patch_report(training, clean_color="green", corrupt_color="green", clean_shape="square", corrupt_shape="circle", position="bottom", question_type="shape", stage="cache")
tests.check_patch_report(color_patch)
tests.check_patch_report(shape_patch)
display(pd.DataFrame(color_patch["rows"]))


<details>
        <summary>Expected output</summary>

        Color and shape object patches flip; background and same-size random patches preserve; full-sequence patches match the corrupt run.

        </details>

        <details>
        <summary>Help - locate the invariant</summary>

        Cache the corrupt activation at the selected stage, then pass those rows as the clean forward's `patch_spec`.

        </details>

        <details>
        <summary>Solution</summary>

        ```python
        def build_single_example(
    *,
    color: str,
    shape: str,
    position: str,
    question_type: str,
    style: str = HELDOUT_STYLE,
    split: str = "patch",
) -> VQABatch:
    """Build a one-example VQA batch for intervention experiments."""

    image, bbox = render_controlled_scene(color, shape, position, style=style)
    answer = color if question_type == "color" else shape
    counterfactual = (
        _next_value(color, COLORS) if question_type == "color" else _next_value(shape, SHAPES)
    )
    example = VQAExample(
        image_id=f"{split}_{style}_{position}_{color}_{shape}_{question_type}",
        color=color,
        shape=shape,
        style=style,
        position=position,
        bbox=bbox,
        question_type=question_type,
        question="What color is the object?"
        if question_type == "color"
        else "What shape is the object?",
        answer=answer,
        counterfactual_answer=counterfactual,
        split=split,
    )
    return VQABatch(
        examples=(example,),
        images=image.unsqueeze(0),
        question_ids=t.tensor([QUESTION_TO_ID[question_type]], dtype=t.long),
        labels=t.tensor([ANSWER_TO_ID[answer]], dtype=t.long),
        counterfactual_labels=t.tensor([ANSWER_TO_ID[counterfactual]], dtype=t.long),
    )


def _patch_replacement_for_stage(
    model: MiniVLM,
    corrupt_cache: t.Tensor,
    question_ids: t.Tensor,
    patch_indices: tuple[int, ...],
    stage: str,
) -> t.Tensor:
    if stage == "cache":
        return corrupt_cache[:, patch_indices]
    with t.no_grad():
        _, caches = model(corrupt_cache, question_ids, return_cache=True)
    return caches[stage][:, patch_indices]


def forward_with_visual_patch(
    model: MiniVLM,
    clean_cache: t.Tensor,
    corrupt_cache: t.Tensor,
    question_ids: t.Tensor,
    patch_indices: tuple[int, ...],
    *,
    stage: str = "cache",
) -> t.Tensor:
    """Run clean example while replacing selected clean activations by corrupt ones."""

    replacement = _patch_replacement_for_stage(
        model,
        corrupt_cache,
        question_ids,
        patch_indices,
        stage,
    )
    logits = model(
        clean_cache,
        question_ids,
        patch_spec={"stage": stage, "indices": patch_indices, "values": replacement},
    )
    assert isinstance(logits, t.Tensor)
    return logits


def _margin_float(logits: t.Tensor, target: int, counterfactual: int) -> float:
    return float((logits[:, target] - logits[:, counterfactual]).detach().cpu().item())


def trained_patch_report(
    training: MiniVLMTrainingResult,
    *,
    clean_color: str = "red",
    corrupt_color: str = "blue",
    clean_shape: str = "square",
    corrupt_shape: str = "square",
    position: str = "top",
    question_type: str = "color",
    stage: str = "cache",
) -> dict[str, object]:
    """Run object/background/random patching for one controlled VQA pair."""

    model = training.model
    encoder = training.encoder
    clean = build_single_example(
        color=clean_color,
        shape=clean_shape,
        position=position,
        question_type=question_type,
    )
    corrupt = build_single_example(
        color=corrupt_color,
        shape=corrupt_shape,
        position=position,
        question_type=question_type,
    )
    device = next(model.parameters()).device
    clean_cache = encode_visual_token_cache(encoder, clean.images, device=device)
    corrupt_cache = encode_visual_token_cache(encoder, corrupt.images, device=device)
    question_ids = clean.question_ids.to(device)
    target = int(clean.labels.item())
    counterfactual = int(corrupt.labels.item())
    object_indices = patch_indices_from_bbox(clean.examples[0].bbox)
    background_indices = same_size_random_region_indices(
        object_indices,
        num_tokens=encoder.num_tokens,
        seed=0,
    )
    random_indices = same_size_random_region_indices(
        object_indices,
        num_tokens=encoder.num_tokens,
        seed=1,
    )
    full_indices = tuple(range(encoder.num_tokens))

    with t.no_grad():
        clean_logits = model(clean_cache, question_ids)
        corrupt_logits = model(corrupt_cache, question_ids)
        object_logits = forward_with_visual_patch(
            model,
            clean_cache,
            corrupt_cache,
            question_ids,
            object_indices,
            stage=stage,
        )
        background_logits = forward_with_visual_patch(
            model,
            clean_cache,
            corrupt_cache,
            question_ids,
            background_indices,
            stage=stage,
        )
        random_region_logits = forward_with_visual_patch(
            model,
            clean_cache,
            corrupt_cache,
            question_ids,
            random_indices,
            stage=stage,
        )
        full_logits = forward_with_visual_patch(
            model,
            clean_cache,
            corrupt_cache,
            question_ids,
            full_indices,
            stage=stage,
        )
    assert isinstance(clean_logits, t.Tensor)
    assert isinstance(corrupt_logits, t.Tensor)

    rows = {
        "clean": clean_logits,
        "corrupt": corrupt_logits,
        "full_sequence_patch": full_logits,
        "object_patch": object_logits,
        "background_patch": background_logits,
        "same_size_random_region_patch": random_region_logits,
    }
    margin_rows = [
        {
            "condition": condition,
            "target_minus_counterfactual": _margin_float(logits, target, counterfactual),
            "predicted_answer": ANSWER_VOCAB[int(logits.argmax(dim=-1).item())],
        }
        for condition, logits in rows.items()
    ]
    margin_by_condition = {
        row["condition"]: float(row["target_minus_counterfactual"])
        for row in margin_rows
    }
    return {
        "stage": stage,
        "question_type": question_type,
        "clean_example": asdict(clean.examples[0]),
        "corrupt_example": asdict(corrupt.examples[0]),
        "target_answer": clean.examples[0].answer,
        "counterfactual_answer": corrupt.examples[0].answer,
        "object_indices": object_indices,
        "background_indices": background_indices,
        "random_indices": random_indices,
        "rows": margin_rows,
        "clean_margin": margin_by_condition["clean"],
        "corrupt_margin": margin_by_condition["corrupt"],
        "object_patch_margin": margin_by_condition["object_patch"],
        "background_patch_margin": margin_by_condition["background_patch"],
        "random_region_patch_margin": margin_by_condition["same_size_random_region_patch"],
        "full_sequence_patch_margin": margin_by_condition["full_sequence_patch"],
        "object_patch_flips": margin_by_condition["object_patch"] < 0,
        "background_patch_preserves": margin_by_condition["background_patch"] > 0,
        "random_region_preserves": margin_by_condition["same_size_random_region_patch"] > 0,
        "full_sequence_matches_corrupt": abs(
            margin_by_condition["full_sequence_patch"] - margin_by_condition["corrupt"]
        )
        < 1e-4,
    }


def patching_effect_heatmap(
    training: MiniVLMTrainingResult,
    *,
    stages: tuple[str, ...] = ("cache", "projected", "value_0", "value_1"),
) -> dict[str, object]:
    """Patch each visual position and measure the target-margin drop."""

    base = trained_patch_report(training, stage="cache")
    clean = build_single_example(
        color=base["clean_example"]["color"],  # type: ignore[index]
        shape=base["clean_example"]["shape"],  # type: ignore[index]
        position=base["clean_example"]["position"],  # type: ignore[index]
        question_type=base["question_type"],  # type: ignore[arg-type]
    )
    corrupt = build_single_example(
        color=base["corrupt_example"]["color"],  # type: ignore[index]
        shape=base["corrupt_example"]["shape"],  # type: ignore[index]
        position=base["corrupt_example"]["position"],  # type: ignore[index]
        question_type=base["question_type"],  # type: ignore[arg-type]
    )
    model = training.model
    encoder = training.encoder
    device = next(model.parameters()).device
    clean_cache = encode_visual_token_cache(encoder, clean.images, device=device)
    corrupt_cache = encode_visual_token_cache(encoder, corrupt.images, device=device)
    question_ids = clean.question_ids.to(device)
    target = int(clean.labels.item())
    counterfactual = int(corrupt.labels.item())
    with t.no_grad():
        clean_logits = model(clean_cache, question_ids)
    assert isinstance(clean_logits, t.Tensor)
    clean_margin = _margin_float(clean_logits, target, counterfactual)

    heatmaps: dict[str, list[list[float]]] = {}
    for stage in stages:
        drops = []
        for token_index in range(encoder.num_tokens):
            with t.no_grad():
                patched_logits = forward_with_visual_patch(
                    model,
                    clean_cache,
                    corrupt_cache,
                    question_ids,
                    (token_index,),
                    stage=stage,
                )
            patched_margin = _margin_float(patched_logits, target, counterfactual)
            drops.append(clean_margin - patched_margin)
        grid = t.tensor(drops).reshape(PATCH_GRID, PATCH_GRID)
        heatmaps[stage] = grid.detach().cpu().tolist()
    return {
        "stages": stages,
        "clean_margin": clean_margin,
        "object_indices": base["object_indices"],
        "heatmaps": heatmaps,
    }
        ```

        </details>

        <details>
        <summary>Interpretation</summary>

        This is causal localization because one answer-bearing attribute changes while question and unrelated scene factors remain fixed.

        </details>


## Signature Result

The final evidence panel is produced from the functions and model you implemented above. It combines the held-out learning curve, modality baselines, exact toy oracle, counterfactual object patches, and all-position causal sweep.

![Mini VLM signature result](../../instructions/assets/mini_vlm_signature_result.png)

![Mini VLM layer-position patching heatmap](../../instructions/assets/mini_vlm_layer_position_patching_heatmap.png)


In [ ]:
heatmap_result = patching_effect_heatmap(training)
signature_result = {
    "train_accuracy": training.train_accuracy,
    "heldout_accuracy": training.heldout_accuracy,
    "baselines": baselines,
    "toy_oracle": toy_report,
    "color_patch": color_patch,
    "shape_patch": shape_patch,
    "patching_heatmap": heatmap_result,
    "loss_curve": list(training.losses),
}
signature_result["accepted"] = (
    training.heldout_accuracy >= 0.95
    and baselines["text_only_accuracy"] <= 0.45
    and baselines["image_only_accuracy"] <= 0.60
    and baselines["random_visual_accuracy"] <= 0.50
    and color_patch["object_patch_flips"]
    and shape_patch["object_patch_flips"]
    and color_patch["background_patch_preserves"]
    and color_patch["random_region_preserves"]
)
reference.save_signature_assets(signature_result, assets_dir)
print("accepted:", signature_result["accepted"])
display(pd.DataFrame([{"condition": "joint", "accuracy": baselines["joint_accuracy"]}, {"condition": "text only", "accuracy": baselines["text_only_accuracy"]}, {"condition": "image only", "accuracy": baselines["image_only_accuracy"]}, {"condition": "shuffled visual", "accuracy": baselines["random_visual_accuracy"]}]))


def run_gpu_test(max_vram_gb: float = 24.0) -> dict[str, object]:
    if not t.cuda.is_available():
        raise RuntimeError("12.3 release verification requires CUDA.")
    t.cuda.reset_peak_memory_stats()
    gpu_training = train_mini_vlm(steps=260, device="cuda")
    result = {"training": gpu_training, "baselines": modality_baseline_report(gpu_training), "color_patch": trained_patch_report(gpu_training), "shape_patch": trained_patch_report(gpu_training, clean_color="green", corrupt_color="green", clean_shape="square", corrupt_shape="circle", position="bottom", question_type="shape")}
    result["peak_vram_gb"] = t.cuda.max_memory_allocated() / 1024**3
    result["within_vram_budget"] = result["peak_vram_gb"] <= max_vram_gb
    return result


def run_full_experiment(max_vram_gb: float = 24.0) -> dict[str, object]:
    return run_gpu_test(max_vram_gb=max_vram_gb)


## Try It Yourself

Change exactly one factor and predict the margin first:

```python
play = trained_patch_report(
    training,
    clean_color="yellow",
    corrupt_color="green",
    clean_shape="triangle",
    corrupt_shape="triangle",
    position="right",
    question_type="color",
    stage="value_1",
)
display(pd.DataFrame(play["rows"]))
```

## Interpreting the Result

- The exact oracle validates patch semantics before neural training.
- Held-out muted rendering tests a real visual distribution shift.
- Text-only failure rejects question priors; image-only failure rejects answer-type ambiguity; shuffled-image failure rejects label leakage.
- Object patch flips plus background/random preservation localize causal evidence.
- Full-sequence parity validates the intervention mechanism.

## Bonus: Hunt an Anomaly

Train on only six color-shape pairs and hold out the other six compositions. The reference architecture often fits training while collapsing near `48%` held-out accuracy. Test whether the failure is caused by connector capacity, correlated training pairs, or the decoder by changing one factor at a time and preserving all controls.

## Limitations

This is a controlled model organism, not evidence about PaliGemma, Qwen-VL, LLaVA, or Molmo. The frozen encoder exposes mean RGB and occupancy, the world contains one object, and the accepted split tests rendering style rather than difficult compositional generalization. These simplifications are explicit so the causal result has ground truth.

## Reading Links

- [Flamingo](https://arxiv.org/abs/2204.14198)
- [BLIP-2](https://arxiv.org/abs/2301.12597)
- [LLaVA](https://arxiv.org/abs/2304.08485)
- [Transformer Circuits activation patching](https://transformer-circuits.pub/2021/framework/index.html)

## Verification Appendix

`verification_report.json` records the post-rewrite CUDA 13.2 run of the visual-prefix causal decoder. It supports the release contract but does not replace the learner-visible training and intervention result.
